# Toric CNOT Logical Error Rates

> Note: This notebook was AI-generated.

This notebook reproduces the toric CNOT experiment from `docs/example_notebooks/toric_cnot_experiment.ipynb`, benchmarks its logical error rate with the `bposd` decoder, and saves both logical-rate data and the distance-3 compilation visual.

In [10]:
import numpy as np
import sinter
import stim

from pathlib import Path

from ldpc import SinterBpOsdDecoder

from epic import (
    AllocCode,
    FreeCode,
    HomologicalMeasurement,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    RotatedSurfaceCode,
    StimLikeNoiseModel,
    ToricCode,
)

In [11]:
def _find_benchmark_base_dir() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in candidates:
        benchmark_dir = root / "docs" / "example_notebooks" / "cnot_benchmark_result"
        if benchmark_dir.exists():
            return benchmark_dir
    # Fallback: save next to the current working directory if the repo layout is not present.
    return Path.cwd().resolve() / "cnot_benchmark_result"


base_dir = _find_benchmark_base_dir()
logical_rates_dir = base_dir / "logical_rates"
visual_dir = base_dir / "visual"
logical_rates_dir.mkdir(parents=True, exist_ok=True)
visual_dir.mkdir(parents=True, exist_ok=True)

PRIMITIVES_CONFIG = {
    "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
    "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
    "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
}

noise_values = np.logspace(-5, -1, 2)
shots = 1
distances = [3, 5, 7]
decoder_name = "bp_osd"
custom_decoders = {decoder_name: SinterBpOsdDecoder()}


def nearest_odd(distance: int) -> int:
    distance = max(3, int(distance))
    return distance if distance % 2 == 1 else distance + 1


def logical_varnames(code, labels: dict[int, str], aux_prefix: str) -> list[str]:
    names = [f"{aux_prefix}_{i}" for i in range(code.k)]
    for logical_index, label in labels.items():
        names[logical_index] = label
    return names


def readout_tag(code_varname: str, code, logical_index: int) -> str:
    logical_name = code.logical_qubits[logical_index].name
    return f"readout_LZ_{code_varname}_{logical_name}"


def compile_toric_cnot_experiment(distance: int, visual_output_path: Path | None = None):
    data_code = ToricCode.from_distance(
        distance=distance,
        code_name=f"toric_data_d{distance}",
        system_coordinate=(0, 0),
    )
    ancilla_distance = nearest_odd(distance)
    ancilla_code = RotatedSurfaceCode.from_distance(
        distance=ancilla_distance,
        code_name=f"ancilla_rsc_d{ancilla_distance}_for_toric_d{distance}",
        system_coordinate=(0, 1),
    )

    shared_var = "data_patch"
    ancilla_var = "ancilla_patch"
    control_label = "control"
    target_label = "target"
    ancilla_label = "ancilla"

    program = [
        AllocCode(
            target_code=data_code,
            code_varname=shared_var,
            logical_qubits_varnames=logical_varnames(
                data_code,
                {0: control_label, 1: target_label},
                aux_prefix=f"{data_code.name}_aux",
            ),
        ),
        AllocCode(
            target_code=ancilla_code,
            code_varname=ancilla_var,
            logical_qubits_varnames=logical_varnames(
                ancilla_code,
                {0: ancilla_label},
                aux_prefix=f"{ancilla_code.name}_aux",
            ),
        ),
        InitCode(targets=[shared_var], initial_state=PauliEigenState.Z_plus, tag=f"init_{shared_var}"),
        InitCode(targets=[ancilla_var], initial_state=PauliEigenState.X_plus, tag=f"init_{ancilla_var}"),
        HomologicalMeasurement(
            targets=[control_label, ancilla_label],
            product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
            tag="MZZ_ca",
        ),
        HomologicalMeasurement(
            targets=[ancilla_label, target_label],
            product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
            tag="MXX_at",
        ),
        ReadoutCode(targets=[shared_var], tag=f"LZ_{shared_var}"),
        ReadoutCode(targets=[ancilla_var], tag=f"LZ_{ancilla_var}"),
        FreeCode(code_varname=shared_var),
        FreeCode(code_varname=ancilla_var),
    ]

    compiler = QECCompiler(
        config={
            "objective_distance": int(distance),
            "primitives": PRIMITIVES_CONFIG,
        }
    )
    compiled_program = compiler.compile(
        program,
        visual_output_path=visual_output_path,
        show_progress=False,
    )
    stim_observables = [
        [readout_tag(shared_var, data_code, 0)],
        [
            "hm_PP_MZZ_ca",
            readout_tag(ancilla_var, ancilla_code, 0),
            readout_tag(shared_var, data_code, 1),
        ],
    ]
    return compiled_program, stim_observables

In [ ]:
saved_results = []

for distance in distances:
    print()
    print(
        f"Starting distance {distance} benchmark with {len(noise_values)} noise points, "
        f"{shots} shots per point, decoder={decoder_name}"
    )

    visual_output_path = (
        visual_dir / "toric_cnot_d3_compilation_visual.pdf" if distance == 3 else None
    )
    print("  Compiling toric CNOT experiment...")
    compiled_program, stim_observables = compile_toric_cnot_experiment(
        distance=distance,
        visual_output_path=visual_output_path,
    )

    tasks = []
    for task_index, physical_noise in enumerate(noise_values, start=1):
        print(f"  Preparing task {task_index}/{len(noise_values)} at p={physical_noise:.2e}")
        noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
            after_clifford_depolarization=float(physical_noise),
            after_reset_flip_probability=float(physical_noise),
            before_measure_flip_probability=float(physical_noise),
            before_round_data_depolarization=float(physical_noise),
        )
        stim_program = compiled_program.to_stim_program(
            stim_observables,
            noise_model=noise_model,
        )
        print(len(stim.Circuit(stim_program).shortest_graphlike_error(ignore_ungraphlike_errors=True)))
        tasks.append(
            sinter.Task(
                circuit=stim.Circuit(stim_program),
                json_metadata={"d": distance, "p": float(physical_noise)},
            )
        )

    print("  Running sinter.collect...")
    collected_stats = sinter.collect(
        num_workers=4,
        tasks=tasks,
        decoders=[decoder_name],
        custom_decoders=custom_decoders,
        max_shots=shots,
        print_progress=True,
    )
    stats_by_noise = {float(stats.json_metadata["p"]): stats for stats in collected_stats}
    logical_error_rates = np.array(
        [
            stats_by_noise[float(physical_noise)].errors
            / stats_by_noise[float(physical_noise)].shots
            for physical_noise in noise_values
        ]
    )
    result_title = f"EPIC toric CNOT logical error rate (d={distance})"
    result_description = (
        f"Logical failure probability for the toric CNOT experiment at distance {distance}. "
        "A shot is counted as a logical fail when either measured logical observable is 1. "
        "Noise uses EPIC StimLikeNoiseModel with matched p for reset flip, Clifford depolarization, "
        "pre-measurement flip, and round-data depolarization channels. Decoding uses BP+OSD."
    )
    results_path = logical_rates_dir / f"toric_cnot_d{distance}_rates.npz"
    np.savez_compressed(
        results_path,
        distance=np.array([distance], dtype=np.int64),
        noise_values=noise_values,
        logical_error_rates=logical_error_rates,
        shots=np.array([shots], dtype=np.int64),
        decoder=np.array([decoder_name]),
        title=np.array([result_title]),
        description=np.array([result_description]),
    )

    saved_results.append(
        {
            "distance": distance,
            "results_path": str(results_path),
            "visual_output_path": str(visual_output_path) if visual_output_path is not None else None,
            "logical_error_rates": logical_error_rates.tolist(),
        }
    )

    print(f"Saved distance {distance} rates to: {results_path.name}")
    if visual_output_path is not None:
        print(f"Saved distance {distance} compilation visual to: {visual_output_path.name}")
    for physical_noise, logical_error_rate in zip(noise_values, logical_error_rates):
        print(f"  p={physical_noise:.2e} -> P_L={logical_error_rate:.6f}")

saved_results